# Phase 1 — Word2Vec Text Representation Experiment

This notebook evaluates **Word2Vec** as the text representation for Software Requirement Prioritization using **LightGBM Ranker** as the fixed baseline ranking model.

**Objective:** Determine whether Word2Vec produces a strong feature representation for requirement prioritization.

**Model:** Pre-computed Word2Vec embeddings (pre-computed in master dataset)

**Ranker:** LightGBM (default parameters, no hyperparameter tuning)


In [1]:
%pip install mlflow lightgbm joblib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import logging
import os
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
    sys.stderr.reconfigure(encoding='utf-8')
import json
import joblib
import warnings

try:
    from IPython.display import display
except ImportError:
    display = print

import mlflow
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import lightgbm as lgb
from scipy.stats import spearmanr, kendalltau
from sklearn.metrics import ndcg_score, average_precision_score  # <-- Tambah average_precision_score di sini
from sklearn.model_selection import GroupShuffleSplit

# Setup MLflow untuk Word2Vec
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri("https://mlflow.smbgarasibmw.my.id/")
mlflow.set_experiment("Phase_1_w2v_Text_Representation")  # <-- Ganti nama eksperimen ke w2v

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

In [3]:
DATASET_PATH = "dataset/Dataset_EDA_Word2Vec.csv"  
OUTPUT_DIR = "outputs/phase1_word2vec"
RANDOM_STATE = 42

TRAIN_GROUP_RATIO = 0.8 

os.makedirs(OUTPUT_DIR, exist_ok=True)
logger.info(f"Output directory initialized at: {OUTPUT_DIR}")

2026-08-13 09:44:27,065 - INFO - Output directory initialized at: outputs/phase1_word2vec


## STEP 1 — Load Dataset

Load the master dataset from `dataset/Dataset_EDA_Word2Vec.csv`. This is the source of truth containing all original metadata and pre-computed Word2Vec embeddings.


In [4]:
# ==========================================
# STEP 1 — Load Dataset & Validation
# ==========================================
logger.info(f"Loading dataset from {DATASET_PATH}...")
df = pd.read_csv(DATASET_PATH)
logger.info(f"Dataset shape: {df.shape}")

# 1. Detect requirement text column otomatis
text_keywords = ["requirement", "text", "description", "sentence", "story", "content"]
text_cols = [c for c in df.columns if any(k in c.lower() for k in text_keywords)]
if text_cols:
    logger.info(f"Detected requirement text column(s): {text_cols}")
    nama_kolom_teks = text_cols[0]  # Kunci nama kolom untuk dipanggil di sel berikutnya
else:
    raise ValueError("No text column found for Word2Vec pipeline!") # <-- SESUAIKAN TEKS

# 2. Validate required columns bisnis & ranking
required_cols = {"id", "project_id", "type", "value", "effort", "risk", "stakeholder_priority", "rank"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

logger.info("All required columns present for Word2Vec ranking pipeline.") # <-- SESUAIKAN TEKS

# 3. CRITICAL SORTING: Wajib diurutkan berdasarkan project_id (sebagai query group)
df = df.sort_values(by="project_id").reset_index(drop=True)
logger.info("Dataset sorted by 'project_id' for ranking context.")

display(df.head(3))

2026-08-13 09:44:27,073 - INFO - Loading dataset from dataset/Dataset_EDA_Word2Vec.csv...
2026-08-13 09:44:27,089 - INFO - Dataset shape: (698, 110)
2026-08-13 09:44:27,089 - INFO - Detected requirement text column(s): ['cleaned_requirement_text']
2026-08-13 09:44:27,090 - INFO - All required columns present for Word2Vec ranking pipeline.
2026-08-13 09:44:27,092 - INFO - Dataset sorted by 'project_id' for ranking context.


,id,project_id,type,value,effort,risk,stakeholder_priority,priority_score,rank,cleaned_requirement_text,...,w2v_90,w2v_91,w2v_92,w2v_93,w2v_94,w2v_95,w2v_96,w2v_97,w2v_98,w2v_99
0,REQ-01,P1,FR,-0.848439,-1.015393,-1.393797,-0.8401,1.3,18,head admin finance edit delete spare part cate...,...,0.002003,0.001158,0.001833,0.000212,0.001036,0.004645,-0.002447,-0.003290,0.003482,-0.000804
1,REQ-505,P1,NFR,-0.848439,0.165847,-0.550024,-0.8401,1.4,16,cosign note record date time signature,...,0.004677,-0.001978,0.002879,-0.002644,0.002841,0.004396,0.006017,-0.003479,0.002732,-0.001574
2,REQ-506,P1,NFR,0.444597,0.165847,1.981294,-0.8401,1.2,25,record display identity addended correct note ...,...,0.003058,0.002929,0.000090,-0.001365,0.003728,0.003701,0.003319,-0.002702,0.003674,0.001436


## STEP 2 — Data Validation

Perform validation checks: missing values, duplicates, invalid project IDs, invalid rank values, and data types.


In [5]:
# ==========================================
# STEP 2 — Data Validation
# ==========================================
validation_report = {}

# Missing values
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]
validation_report["missing_values"] = len(missing_cols)
if len(missing_cols) > 0:
    logger.warning(f"Columns with missing values:\n{missing_cols}")
else:
    logger.info("No missing values found.")

# Duplicate rows
dup_rows = df.duplicated().sum()
validation_report["duplicate_rows"] = dup_rows
if dup_rows > 0:
    logger.warning(f"Duplicate rows: {dup_rows}")
else:
    logger.info("No duplicate rows found.")

# Duplicate requirements (by id)
dup_ids = df["id"].duplicated().sum()
validation_report["duplicate_ids"] = dup_ids
if dup_ids > 0:
    logger.warning(f"Duplicate requirement IDs: {dup_ids}")
else:
    logger.info("No duplicate IDs found.")

# Invalid project_id
invalid_pid = df["project_id"].isnull().sum() + (df["project_id"].astype(str).str.strip() == "").sum()
validation_report["invalid_project_ids"] = int(invalid_pid)
if invalid_pid > 0:
    logger.warning(f"Invalid project IDs: {invalid_pid}")
else:
    logger.info("All project IDs are valid.")

# Invalid rank (should be numeric, non-negative)
invalid_rank = (~pd.to_numeric(df["rank"], errors="coerce").notna()).sum()
validation_report["invalid_rank"] = int(invalid_rank)
if invalid_rank > 0:
    logger.warning(f"Invalid rank values: {invalid_rank}")
else:
    logger.info("All rank values are valid.")

# Data types
validation_report["dtypes"] = {c: str(dt) for c, dt in df.dtypes.items()}

logger.info("=== Validation Report ===")
for k, v in validation_report.items():
    logger.info(f"  {k}: {v}")

2026-08-13 09:44:27,115 - INFO - No missing values found.
2026-08-13 09:44:27,124 - INFO - No duplicate rows found.
2026-08-13 09:44:27,124 - INFO - No duplicate IDs found.
2026-08-13 09:44:27,128 - INFO - All project IDs are valid.
2026-08-13 09:44:27,128 - INFO - All rank values are valid.
2026-08-13 09:44:27,129 - INFO - === Validation Report ===
2026-08-13 09:44:27,131 - INFO -   missing_values: 0
2026-08-13 09:44:27,131 - INFO -   duplicate_rows: 0
2026-08-13 09:44:27,131 - INFO -   duplicate_ids: 0
2026-08-13 09:44:27,131 - INFO -   invalid_project_ids: 0
2026-08-13 09:44:27,131 - INFO -   invalid_rank: 0
2026-08-13 09:44:27,132 - INFO -   dtypes: {'id': 'object', 'project_id': 'object', 'type': 'object', 'value': 'float64', 'effort': 'float64', 'risk': 'float64', 'stakeholder_priority': 'float64', 'priority_score': 'float64', 'rank': 'int64', 'cleaned_requirement_text': 'object', 'w2v_0': 'float64', 'w2v_1': 'float64', 'w2v_2': 'float64', 'w2v_3': 'float64', 'w2v_4': 'float64', 

## STEP 3 — Text Representation (Word Embedding) & MLflow Run Start

Initialize MLflow run and prepare text representation features.


In [6]:
logger.info("Preparing pre-computed Word2Vec dataset...")

# Filter semua kolom yang namanya diawali dengan 'w2v_'
w2v_cols = [c for c in df.columns if c.startswith("w2v_")]
X_w2v = df[w2v_cols].values  # Langsung jadi matriks fitur teks Word2Vec

logger.info(f"Loaded pre-computed Word2Vec matrix. Shape: {X_w2v.shape}")


2026-08-13 09:44:27,138 - INFO - Preparing pre-computed Word2Vec dataset...
2026-08-13 09:44:27,140 - INFO - Loaded pre-computed Word2Vec matrix. Shape: (698, 100)


## STEP 4 — Feature Fusion

Fuse numerical business features (value, effort, risk, stakeholder), Word2Vec text representation features, and one-hot encoded categorical features into a single dataset.


In [7]:
# ==========================================
# STEP 4 — Feature Fusion (Word2Vec Version)
# ==========================================
num_features = ["value", "effort", "risk", "stakeholder_priority"]
logger.info(f"Numerical business features to fuse: {num_features}")

# 1. Ambil nilai numerik dari DataFrame menjadi NumPy Array
X_num = df[num_features].values

# 2. Fitur teks Word2Vec langsung diambil dari STEP 3 (Sudah berbentuk Dense Array)
X_text_dense = X_w2v 

# 3. Proses One-Hot Encoding langsung di sini (Biar mandiri & gak ketergantungan sel lain)
logger.info("Performing One-Hot Encoding on 'type' column...")
encoded_type_df = pd.get_dummies(df["type"], prefix="type")
X_encoded = encoded_type_df.values
logger.info(f"One-hot encoded type columns: {list(encoded_type_df.columns)}")

# 4. FUSI DATA: Gabungkan fitur bisnis, embedding Word2Vec, dan category encoding secara horizontal
X_fused = np.hstack((X_num, X_text_dense, X_encoded))

# 5. Catat dimensi matriks final ke MLflow
mlflow.log_param("total_fused_features", X_fused.shape[1])

logger.info(f"Fusion completed:")
logger.info(f" - Numerical dimension: {X_num.shape[1]}")
logger.info(f" - Word2Vec text dimension: {X_text_dense.shape[1]}")
logger.info(f" - Encoded categorical dimension: {X_encoded.shape[1]}")
logger.info(f" - Final Fused Feature Matrix shape (X): {X_fused.shape}")

# Mengintip beberapa baris pertama hasil fusi (fitur numerik bisnis berada di 4 kolom terdepan)
print("\nFirst 3 rows of final feature matrix (X) looks like:")
print(X_fused[:3, :8]) # Intip 4 fitur numerik + 4 dimensi W2V pertama

2026-08-13 09:44:27,149 - INFO - Numerical business features to fuse: ['value', 'effort', 'risk', 'stakeholder_priority']
2026-08-13 09:44:27,150 - INFO - Performing One-Hot Encoding on 'type' column...
2026-08-13 09:44:27,152 - INFO - One-hot encoded type columns: ['type_FR', 'type_NFR']
2026-08-13 09:44:27,442 - INFO - Fusion completed:
2026-08-13 09:44:27,444 - INFO -  - Numerical dimension: 4
2026-08-13 09:44:27,445 - INFO -  - Word2Vec text dimension: 100
2026-08-13 09:44:27,445 - INFO -  - Encoded categorical dimension: 2
2026-08-13 09:44:27,445 - INFO -  - Final Fused Feature Matrix shape (X): (698, 106)



First 3 rows of final feature matrix (X) looks like:
[[-8.48439158e-01 -1.01539269e+00 -1.39379651e+00 -8.40099608e-01
  -5.59904000e-04  2.10441280e-03 -3.51560700e-04  5.54474830e-05]
 [-8.48439158e-01  1.65847473e-01 -5.50023776e-01 -8.40099608e-01
  -3.53739880e-03  1.01571400e-03  1.55588430e-03 -4.35219200e-03]
 [ 4.44596939e-01  1.65847473e-01  1.98129444e+00 -8.40099608e-01
  -3.75702230e-03  1.02912540e-03 -4.95557760e-05 -1.62583000e-03]]


## STEP 5 — Query Group Construction

Prepare the dataset for Learning to Rank:

- **Query/group identifier:** `project_id`

- **Ranking label:** `rank`

- Verify every project contains multiple requirements.


In [8]:
logger.info("Transforming ranks into LambdaRank-compatible labels...")

# 1. Transformasi rank ke skor relevansi (0 = terburuk, N-1 = terbaik)
df["label"] = df.groupby("project_id")["rank"].transform(
    lambda x: x.rank(method="dense", ascending=False).astype(int) - 1
)

# 2. Definisikan nilai target y akhir dari kolom label baru
y = df["label"].values
query_ids = df["project_id"]

# 3. Hitung distribusi ukuran tiap kelompok proyek (Query Group)
group_sizes = query_ids.value_counts()
num_groups = len(group_sizes)

logger.info(f"Number of query groups (projects): {num_groups}")
logger.info(f"Group size stats:\n{group_sizes.describe()}")

# Peringatan untuk proyek yang isinya cuma 1 kebutuhan
single_req_groups = (group_sizes == 1).sum()
if single_req_groups > 0:
    logger.warning(f"Groups with only 1 requirement: {single_req_groups} — these cannot be ranked")

# 4. Kunci ukuran grup untuk kebutuhan parameter training LightGBM Ranker
group_counts = group_sizes.sort_index().values  # Urut berdasarkan project_id
logger.info(f"Group sizes array for LightGBM (first 10): {group_counts[:10]}...")

# 5. Log metrik distribusi grup dan label ke MLflow
mlflow.log_param("num_query_groups", num_groups)
mlflow.log_param("label_min_value", int(y.min()))
mlflow.log_param("label_max_value", int(y.max()))

logger.info(f"Label validation -> Range: [{y.min()}, {y.max()}], Unique labels: {sorted(np.unique(y))[:20]}...")

2026-08-13 09:44:27,456 - INFO - Transforming ranks into LambdaRank-compatible labels...
2026-08-13 09:44:27,462 - INFO - Number of query groups (projects): 15
2026-08-13 09:44:27,464 - INFO - Group size stats:
count     15.000000
mean      46.533333
std       64.790725
min        3.000000
25%       10.000000
50%       18.000000
75%       49.500000
max      231.000000
Name: count, dtype: float64
2026-08-13 09:44:27,466 - INFO - Group sizes array for LightGBM (first 10): [ 47   8  18  10  19   9   3 231  99 146]...
2026-08-13 09:44:27,705 - INFO - Label validation -> Range: [0, 20], Unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]...


## STEP 6 — Train/Test Split

Use `GroupShuffleSplit` (or `GroupAwareSplit`) to ensure the same project never appears in both train and test sets. Fixed `random_state=42` for reproducibility.


In [9]:
# ==========================================
# STEP 8 — Group-Aware Train-Test Split
# ==========================================
# Definisikan ulang variabel config yang hilang akibat restart kernel
TEST_SIZE = 0.2      # Sesuaikan dengan ratio lu (misal 0.2 atau 0.3)
RANDOM_STATE = 42    # Sesuaikan dengan random state awal lu

logger.info("Performing group-aware train-test split using GroupShuffleSplit...")

# 1. Definisikan array groups berdasarkan project_id
groups = df["project_id"].values

# 2. Inisialisasi GroupShuffleSplit memakai parameter dari global config
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X_fused, y, groups=groups))

# 3. Slice matriks X_fused (Wajib slice langsung, JANGAN pakai .iloc karena ini NumPy Array!)
X_train = X_fused[train_idx]
X_test = X_fused[test_idx]

# 4. Slice target label dan kelompok data
y_train = y[train_idx]
y_test = y[test_idx]
groups_train = groups[train_idx]
groups_test = groups[test_idx]

# 5. Hitung ulang jumlah baris per grup kueri untuk parameter input LightGBM Ranker
train_group_counts = pd.Series(groups_train).value_counts().sort_index().values
test_group_counts = pd.Series(groups_test).value_counts().sort_index().values

logger.info(f"Train size: {len(X_train)} rows, Test size: {len(X_test)} rows")
logger.info(f"Train groups: {len(train_group_counts)} projects, Test groups: {len(test_group_counts)} projects")

# 6. Validasi ketat: Pastikan tidak ada proyek yang tumpang tindih
train_projects = set(np.unique(groups_train))
test_projects = set(np.unique(groups_test))
overlap = train_projects & test_projects
if overlap:
    raise ValueError(f"Project overlap between train and test: {overlap}")
logger.info("No project overlap between train and test sets. Split is clean.")



2026-08-13 09:44:27,714 - INFO - Performing group-aware train-test split using GroupShuffleSplit...
2026-08-13 09:44:27,733 - INFO - Train size: 453 rows, Test size: 245 rows
2026-08-13 09:44:27,733 - INFO - Train groups: 12 projects, Test groups: 3 projects
2026-08-13 09:44:27,733 - INFO - No project overlap between train and test sets. Split is clean.


## STEP 7 — LightGBM Ranker Training

Train a baseline LightGBM Ranker with reasonable default parameters. No hyperparameter tuning — the goal is to evaluate Word2Vec representation quality, not to optimize the ranker.


In [10]:
# 1. Hitung jumlah maksimum num_leaves secara dinamis berdasarkan ukuran grup terbesar
max_group_size = df.groupby("project_id").size().max()
num_leaves = min(max_group_size, 255)

# 2. Inisialisasi LGBMRanker
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    boosting_type="gbdt",
    n_estimators=100,
    num_leaves=num_leaves,
    learning_rate=0.1,
    min_child_samples=10,
    random_state=RANDOM_STATE,
    verbose=-1
)

logger.info("Training LightGBM Ranker for Word2Vec...")
# 3. Fit model menggunakan array group counts yang sudah kita split di sel sebelumnya
ranker.fit(
    X_train, y_train,
    group=train_group_counts,
    eval_set=[(X_test, y_test)],
    eval_group=[test_group_counts],
    eval_metric=["ndcg"],
    callbacks=[lgb.log_evaluation(0)]
)
logger.info("Training complete.")


# ==========================================
# REKONSTRUKSI NAMA FITUR UNTUK IMPORTANCE (WORD2VEC FIXED)
# ==========================================
# Ambil semua nama kolom Word2Vec langsung dari DataFrame karena tidak ada seleksi fitur
w2v_cols = [c for c in df.columns if c.startswith("w2v_")]

# Satukan semua nama fitur sesuai urutan horizontal stack (hstack) di STEP 4 kemarin:
# Fitur Bisnis + Fitur Teks Word2Vec + Fitur Kategori (One-Hot)
all_feature_names = num_features + w2v_cols + list(encoded_type_df.columns)

# Tampilkan 10 fitur paling berpengaruh tanpa error
feature_imp_series = pd.Series(ranker.feature_importances_, index=all_feature_names)
logger.info(f"Feature importances (top 10):\n{feature_imp_series.sort_values(ascending=False).head(10)}")

2026-08-13 09:44:27,744 - INFO - Training LightGBM Ranker for Word2Vec...
2026-08-13 09:44:29,169 - INFO - Training complete.
2026-08-13 09:44:29,169 - INFO - Feature importances (top 10):
effort    179
value     157
risk      130
w2v_0      89
w2v_77     60
w2v_32     59
w2v_63     55
w2v_53     53
w2v_46     51
w2v_25     51
dtype: int32


## STEP 8 — Model Evaluation

Evaluate using:

- **NDCG@5, NDCG@10** — Normalized Discounted Cumulative Gain

- **MAP** — Mean Average Precision

- **Spearman Rank Correlation**

- **Kendall Tau**


In [11]:
from sklearn.metrics import average_precision_score, ndcg_score

logger.info("Generating predictions on test set...")
# 1. Prediksi skor relevansi menggunakan model ranker
y_pred = ranker.predict(X_test)

test_project_ids = groups_test
ndcg5_scores = []
ndcg10_scores = []
map_per_group = []

logger.info("Computing ranking metrics per project group...")
# 2. Iterasi per kelompok proyek untuk menghitung NDCG dan MAP
for pid in np.unique(test_project_ids):
    mask = test_project_ids == pid
    y_true_group = y_test[mask]
    y_pred_group = y_pred[mask]
    n = len(y_true_group)

    # Lewati proyek yang isinya cuma 1 kebutuhan karena tidak bisa diranking
    if n <= 1:
        continue

    # NDCG@5 (Diperbaiki: Proyek kecil tetap dihitung dengan k dinamis)
    k5 = min(5, n)
    ndcg5_scores.append(ndcg_score(y_true_group.reshape(1, -1), y_pred_group.reshape(1, -1), k=k5))
    
    # NDCG@10 (Diperbaiki: Proyek kecil tetap dihitung dengan k dinamis)
    k10 = min(10, n)
    ndcg10_scores.append(ndcg_score(y_true_group.reshape(1, -1), y_pred_group.reshape(1, -1), k=k10))

    # MAP (Binarisasi: top half labels dianggap relevan)
    if len(np.unique(y_true_group)) > 1:
        threshold = y_true_group.max() * 0.5
        y_bin = (y_true_group >= threshold).astype(int)
        if y_bin.sum() > 0 and y_bin.sum() < len(y_bin):
            map_per_group.append(average_precision_score(y_bin, y_pred_group))

# Hitung rata-rata metrik
ndcg5 = float(np.mean(ndcg5_scores)) if ndcg5_scores else 0.0
ndcg10 = float(np.mean(ndcg10_scores)) if ndcg10_scores else 0.0
map_score = float(np.mean(map_per_group)) if map_per_group else 0.0

# 3. Hitung korelasi ranking global (Spearman & Kendall Tau)
spearman_corr, spearman_p = spearmanr(y_test, y_pred)
kendall_corr, kendall_p = kendalltau(y_test, y_pred)

# Group semua metrik ke dalam dictionary
metrics = {
    "NDCG_at_5": round(float(ndcg5), 6),
    "NDCG_at_10": round(float(ndcg10), 6),
    "MAP": round(float(map_score), 6),
    "Spearman": round(float(spearman_corr), 6),
    "Spearman_pvalue": float(spearman_p),
    "KendallTau": round(float(kendall_corr), 6),
    "KendallTau_pvalue": float(kendall_p)
}

logger.info("=== Evaluation Metrics ===")
for k, v in metrics.items():
    logger.info(f"  {k}: {v}")



# Tampilkan tabel metrik di Jupyter Notebook
display(pd.DataFrame([metrics]))

2026-08-13 09:44:29,187 - INFO - Generating predictions on test set...
2026-08-13 09:44:29,189 - INFO - Computing ranking metrics per project group...
2026-08-13 09:44:29,200 - INFO - === Evaluation Metrics ===
2026-08-13 09:44:29,201 - INFO -   NDCG_at_5: 0.941569
2026-08-13 09:44:29,201 - INFO -   NDCG_at_10: 0.964541
2026-08-13 09:44:29,203 - INFO -   MAP: 0.913571
2026-08-13 09:44:29,203 - INFO -   Spearman: 0.730693
2026-08-13 09:44:29,203 - INFO -   Spearman_pvalue: 3.649558511221468e-42
2026-08-13 09:44:29,203 - INFO -   KendallTau: 0.541301
2026-08-13 09:44:29,204 - INFO -   KendallTau_pvalue: 3.688073493489777e-34


,NDCG_at_5,NDCG_at_10,MAP,Spearman,Spearman_pvalue,KendallTau,KendallTau_pvalue
0,0.941569,0.964541,0.913571,0.730693,3.649559e-42,0.541301,3.688073e-34


## STEP 9 — Save Outputs

Save all experiment outputs to `outputs/phase1_word2vec/`:

- Trained LightGBM model

- Generated Word2Vec dataset (for Phase 2 input)

- Evaluation metrics (JSON)

- Predictions

- Feature list


In [12]:
logger.info("Saving training artifacts and generated dataset locally...")

# 1. Save trained model
model_path = os.path.join(OUTPUT_DIR, "lgbm_ranker.pkl")
joblib.dump(ranker, model_path)
logger.info(f"Model saved locally: {model_path}")

# 2. Save generated dataset (Phase 2 input)
dataset_out = pd.DataFrame(X_fused, columns=all_feature_names)
dataset_out["label"] = y
dataset_out["rank"] = df["rank"].values
dataset_out["project_id"] = groups

# UBAH: Ganti nama file dari tfidf_dataset.csv menjadi w2v_dataset.csv
dataset_path = os.path.join(OUTPUT_DIR, "w2v_dataset.csv") 
dataset_out.to_csv(dataset_path, index=False)
logger.info(f"Generated dataset saved locally: {dataset_path} (shape: {dataset_out.shape})")

# 3. Save evaluation metrics
metrics_path = os.path.join(OUTPUT_DIR, "evaluation_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
logger.info(f"Metrics saved locally: {metrics_path}")

# 4. Save predictions
predictions_df = pd.DataFrame({
    "project_id": groups_test,
    "true_label": y_test,
    "predicted_score": y_pred
})
pred_path = os.path.join(OUTPUT_DIR, "predictions.csv")
predictions_df.to_csv(pred_path, index=False)
logger.info(f"Predictions saved locally: {pred_path}")

# 5. Save feature list
features_path = os.path.join(OUTPUT_DIR, "feature_list.txt")
with open(features_path, "w") as f:
    f.write("\n".join(all_feature_names))
logger.info(f"Feature list saved locally: {features_path}")

2026-08-13 09:44:29,213 - INFO - Saving training artifacts and generated dataset locally...
2026-08-13 09:44:29,232 - INFO - Model saved locally: outputs/phase1_word2vec\lgbm_ranker.pkl
2026-08-13 09:44:29,304 - INFO - Generated dataset saved locally: outputs/phase1_word2vec\w2v_dataset.csv (shape: (698, 109))
2026-08-13 09:44:29,414 - INFO - Metrics saved locally: outputs/phase1_word2vec\evaluation_metrics.json
2026-08-13 09:44:29,428 - INFO - Predictions saved locally: outputs/phase1_word2vec\predictions.csv
2026-08-13 09:44:29,429 - INFO - Feature list saved locally: outputs/phase1_word2vec\feature_list.txt


## STEP 10 — MLflow Logging

Track the experiment using MLflow: log parameters, metrics, and artifacts.


In [ ]:
if mlflow.active_run():
    mlflow.end_run()

try:
    with mlflow.start_run(run_name="Precomputed_13-August-2026_Phase-1") as run:
        # Log parameters
        mlflow.log_param("text_representation", "Word2Vec")
        mlflow.log_param("feature_selection_method", "None")
        mlflow.log_param("w2v_features_count", X_w2v.shape[1])
        mlflow.log_param("total_fused_features", X_fused.shape[1])
        mlflow.log_param("train_size", len(X_train))
        mlflow.log_param("test_size", len(X_test))
        mlflow.log_param("num_train_groups", len(train_group_counts))
        mlflow.log_param("num_test_groups", len(test_group_counts))
        mlflow.log_param("random_state", RANDOM_STATE)
        mlflow.log_param("test_size_ratio", globals().get("TEST_SIZE", 0.2))
        mlflow.log_param("ranker_objective", "lambdarank")
        mlflow.log_param("ranker_n_estimators", 100)
        mlflow.log_param("ranker_num_leaves", num_leaves)

        # Log metrics
        mlflow.log_metrics(metrics)

        # Log artifacts
        mlflow.log_artifact(model_path, artifact_path="model")
        mlflow.log_artifact(metrics_path, artifact_path="metrics")
        mlflow.log_artifact(features_path, artifact_path="features")
        mlflow.log_artifact(dataset_path, artifact_path="dataset")
        mlflow.log_artifact(pred_path, artifact_path="predictions")

        logger.info(f"MLflow run ID: {run.info.run_id}")
        logger.info("MLflow experiment logged successfully.")
except Exception as e:
    logger.warning(f"MLflow logging error: {e}. Experiment execution & local artifacts completed successfully.")



🏃 View run crawling-cow-204 at: https://mlflow.smbgarasibmw.my.id/#/experiments/3/runs/a6d0cd7e98334651b2bed6fb4d5a1f8e
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/3


2026-08-13 09:44:31,126 - INFO - MLflow run ID: c04d39cc21ce40a3b278988f284b8b0f
2026-08-13 09:44:31,126 - INFO - MLflow experiment logged successfully.


🏃 View run Word2Vec_LightGBM_Ranker_Experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/3/runs/c04d39cc21ce40a3b278988f284b8b0f
🧪 View experiment at: https://mlflow.smbgarasibmw.my.id/#/experiments/3


## Experiment Summary

### What was done

1. **Dataset:** Loaded master dataset with pre-computed Word2Vec embeddings

2. **Features:** 4 numerical business features (value, effort, risk, stakeholder priority) + Word2Vec embedding dimensions + one-hot encoded type

3. **Model:** LightGBM Ranker (Lambdarank objective, default parameters)

4. **Split:** GroupShuffleSplit (no project overlap between train/test)

5. **Evaluation:** NDCG@5, NDCG@10, MAP, Spearman, Kendall Tau


In [14]:
# 1. Siapkan DataFrame rangkuman metrik agar rapi saat di-display
summary_data = {
    "Metric": ["NDCG@5", "NDCG@10", "MAP", "Spearman rho", "Spearman p-value", "Kendall tau", "Kendall tau p-value"],
    "Value": [
        metrics["NDCG_at_5"],
        metrics["NDCG_at_10"],
        metrics["MAP"],
        metrics["Spearman"],
        metrics["Spearman_pvalue"],
        metrics["KendallTau"],
        metrics["KendallTau_pvalue"]
    ]
}
summary_df = pd.DataFrame(summary_data)

# 2. Cetak log parameter eksperimen Word2Vec ke konsol/terminal
logger.info("=== Experiment Summary ===")
logger.info("Text Representation: Word2Vec")  # <-- UBAH: Set ke Word2Vec
logger.info("Feature Selection Method: None (Skipped)")
logger.info(f"Word2Vec Text Dimension: {X_w2v.shape[1]}")  # <-- UBAH: Panggil X_w2v hasil STEP 3
logger.info(f"Train size: {len(X_train)} rows, Test size: {len(X_test)} rows")
logger.info(f"Train groups: {len(train_group_counts)} projects, Test groups: {len(test_group_counts)} projects")
logger.info(f"Total Final Features (Fused): {X_fused.shape[1]}")
logger.info(f"\n{summary_df.to_string(index=False)}")

# 3. Tampilkan tabel metrik di Jupyter notebook
display(summary_df)

2026-08-13 09:44:31,349 - INFO - === Experiment Summary ===
2026-08-13 09:44:31,350 - INFO - Text Representation: Word2Vec
2026-08-13 09:44:31,350 - INFO - Feature Selection Method: None (Skipped)
2026-08-13 09:44:31,350 - INFO - Word2Vec Text Dimension: 100
2026-08-13 09:44:31,350 - INFO - Train size: 453 rows, Test size: 245 rows
2026-08-13 09:44:31,352 - INFO - Train groups: 12 projects, Test groups: 3 projects
2026-08-13 09:44:31,352 - INFO - Total Final Features (Fused): 106
2026-08-13 09:44:31,353 - INFO - 
             Metric        Value
             NDCG@5 9.415690e-01
            NDCG@10 9.645410e-01
                MAP 9.135710e-01
       Spearman rho 7.306930e-01
   Spearman p-value 3.649559e-42
        Kendall tau 5.413010e-01
Kendall tau p-value 3.688073e-34


,Metric,Value
0,NDCG@5,9.415690e-01
1,NDCG@10,9.645410e-01
2,MAP,9.135710e-01
3,Spearman rho,7.306930e-01
4,Spearman p-value,3.649559e-42
5,Kendall tau,5.413010e-01
6,Kendall tau p-value,3.688073e-34


### Outputs

All artifacts saved to `outputs/phase1_word2vec/`. The generated dataset is ready for Phase 2.


---

*Phase 1 — Word2Vec Text Representation Experiment complete.*
